# 2.BPE 分词器

## 2.1 unicode 标准

In [1]:
ord('s')

115

In [2]:
ord('牛')

29275

In [3]:
chr(115)

's'

In [4]:
chr(29275)

'牛'

#### problem：理解unicode

1.chr(0)返回的什么unicode字符

In [5]:
chr(0)

'\x00'

2.这个字符的字符串表示（__repr__()）与它的打印表示有什么不同

In [6]:
print(chr(0))

 


3.这个字符在文本中是什么样子

In [7]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [8]:
print("this is a test"+ chr(0) + "string")

this is a test string


## 2.2 Unicode 编码

In [9]:
test_string="hello! 孙嘉晨!"
utf8_encoded = test_string.encode("utf-8")

In [10]:
print(utf8_encoded)

b'hello! \xe5\xad\x99\xe5\x98\x89\xe6\x99\xa8!'


In [11]:
#得到encoded string的byte value （0-255的整数）
list(utf8_encoded)

[104,
 101,
 108,
 108,
 111,
 33,
 32,
 229,
 173,
 153,
 229,
 152,
 137,
 230,
 153,
 168,
 33]

In [12]:
#一个字节并不一定对应一个字符
print(len(test_string))
print(len(utf8_encoded))

11
17


In [13]:
print(utf8_encoded.decode("utf-8"))

hello! 孙嘉晨!


#### problem:Unicode 编码

1.相比于 UTF-16 或 UTF-32，为什么我们更倾向于在 UTF-8 编码的字节上训练 tokenizer？比较不同输入字符串在这几种编码下的输出可能会有帮助。

In [77]:
#UTF-8是可变长编码，对ASCII字符只使用1个字节，空间利用率高且兼容广泛。UTF-16与UTF-32使用固定或者更长的字节，导致词汇表更大、数据稀疏，不利于tokenzier学习高效子词单元
text="hello! 孙嘉晨!"
print(text.encode("utf-8"))
print(text.encode("utf-16"))
print(text.encode("utf-32"))

b'hello! \xe5\xad\x99\xe5\x98\x89\xe6\x99\xa8!'
b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00Y[\tVhf!\x00'
b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00!\x00\x00\x00 \x00\x00\x00Y[\x00\x00\tV\x00\x00hf\x00\x00!\x00\x00\x00'


2.考虑下面这个（错误的）函数，其目的是将UTF-8字节串解码为Unicode字符串。为什么这个函数是错误的？提供一个会产生错误结果的输入字节串的例子

In [78]:
def decode_utf8_bytes_to_str_wrong(bytestring:bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [79]:
def decode_utf8_bytes_to_str_wrong(bytestring:bytes):#可以成功的
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

'hello'

In [80]:
decode_utf8_bytes_to_str_wrong("café".encode("utf-8"))#不可以成功的

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 0: unexpected end of data

3.给一个不解码成任何Unicode字符的双字节序列。

In [ ]:
b'\x80\x80'.decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

## 2.3 subword Tokenization

一种介于word-level 跟byte-level 之间折中方案

## 2.4 BPE 分词器 训练

训练分为三部分：词表初始化、预分词、计算 BPE merges

In [19]:
pip install regex


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import regex as re

In [21]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
re.findall(PAT,"some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

In [22]:
max([("A","B"),("A","C"),("B","ZZ"),("BA","A")])

('BA', 'A')

In [23]:
print("="*60)
print("BPE 分词器学习step-by-step ")
print("="*60)

BPE 分词器学习step-by-step 


In [24]:
print("lesson1 理解字节和字符串")
print("-"*40)

lesson1 理解字节和字符串
----------------------------------------


In [25]:
text="hello"
print(f"原始字符串:{text}")
print(f"字符串类型:{type(text)}")

原始字符串:hello
字符串类型:<class 'str'>


In [26]:
#转化为字节
bytes_text=text.encode("utf-8")
print(f"\n编码为字节:{bytes_text}")
print(f"字节类型:{type(bytes_text)}")


编码为字节:b'hello'
字节类型:<class 'bytes'>


In [27]:
#拆分为单个字节
byte_list=[bytes([b]) for b in bytes_text]
print(f"\n拆分成单字节列表:")
for i ,b in enumerate(byte_list):
    print(f" 位置{i}:{b}(值:{ord(b.decode())})")


拆分成单字节列表:
 位置0:b'h'(值:104)
 位置1:b'e'(值:101)
 位置2:b'l'(值:108)
 位置3:b'l'(值:108)
 位置4:b'o'(值:111)


In [28]:
print("\n 关键概念:")
print("- 字符串'hello' -> 字节 b'hello'")
print("- 每个字符对应1个字节（英文）")



 关键概念:
- 字符串'hello' -> 字节 b'hello'
- 每个字符对应1个字节（英文）


In [29]:
print("\n lesson2 统计相邻字节对")
print("-"*40)


 lesson2 统计相邻字节对
----------------------------------------


In [30]:
#准备一个简单的语料
corpus = ["hello","hello","world"]
print(f"语料:{corpus}")

语料:['hello', 'hello', 'world']


In [31]:
#统计pair 频率
from collections import Counter

In [32]:
all_pairs = []
for word in corpus:
    word_bytes = [bytes([b]) for b in word.encode("utf-8")]
    pairs = [(word_bytes[i],word_bytes[i+1])for i in range(len(word_bytes)-1)]
    all_pairs.extend(pairs)

In [33]:
pair_counts= Counter(all_pairs)
print(f"\n所有相邻对及频率:")
for pair ,count in pair_counts.most_common():
    print(f"{pair[0].decode()}+{pair[1].decode()}:{count}次")


所有相邻对及频率:
h+e:2次
e+l:2次
l+l:2次
l+o:2次
w+o:1次
o+r:1次
r+l:1次
l+d:1次


In [34]:
print("BPE 核心思想:")
print(f" -最频繁的对是:{pair_counts.most_common(1)[0][0]}")
print(" -我们应该先合并这个！")

BPE 核心思想:
 -最频繁的对是:(b'h', b'e')
 -我们应该先合并这个！


In [35]:
print("\n\n lesson3 手动模拟一次BPE 合并")
print("-"*40)



 lesson3 手动模拟一次BPE 合并
----------------------------------------


In [36]:
#初始状态
vocab = {i:bytes([i])for i in range(256)}
sequences = [
    tuple(bytes([b]) for b in "hello".encode("utf-8")),
    tuple(bytes([b]) for b in "world".encode("utf-8")),
]

In [37]:
print(f"初始词汇表大小:{len(vocab)}")
print(f"初始序列：")
for seq in sequences:
    print(f" {seq}")

初始词汇表大小:256
初始序列：
 (b'h', b'e', b'l', b'l', b'o')
 (b'w', b'o', b'r', b'l', b'd')


In [38]:
print("\n--- 第1次合并 ---")


--- 第1次合并 ---


In [39]:
#统计所pairs
pair_counts = Counter()
for seq in sequences:
    for i in range(len(seq)-1):
        pair = (seq[i],seq[i+1])
        pair_counts[pair]+=1

In [40]:
print("Pair 频率统计:")
for pair,count in pair_counts.most_common(5):
    print(f" {pair[0].decode()}+{pair[1].decode()}:{count}次")


Pair 频率统计:
 h+e:1次
 e+l:1次
 l+l:1次
 l+o:1次
 w+o:1次


In [41]:
#选择最频繁的pair
best_pair = max(pair_counts,key=pair_counts.get)
print(f" 选择最频繁的 pair:{best_pair[0].decode()}+{best_pair[1].decode()}")

 选择最频繁的 pair:h+e


In [42]:
#创建新的token
new_token = best_pair[0]+best_pair[1]
new_id = len(vocab)
vocab[new_id]=new_token

In [43]:
print(f"创建新 token : ID ={new_id},值={new_token}")

创建新 token : ID =256,值=b'he'


In [44]:
#在序列中合并
def merge_in_sequences(seq,pair,new_token):
    new_seq=[]
    i=0
    while i<len(seq):
        if i < len(seq)-1 and seq[i] == pair[0] and seq[i+1] == pair[1]:
            new_seq.append(new_token)
            i+=2
        else:
            new_seq.append(seq[i])
            i+=1
    return tuple(new_seq)

In [45]:
sequences=[merge_in_sequences(seq,best_pair,new_token) for seq in sequences]

In [46]:
print(f"n 合并后序列:")
for seq in sequences:
    print(f"{seq}")

n 合并后序列:
(b'he', b'l', b'l', b'o')
(b'w', b'o', b'r', b'l', b'd')


In [47]:
print(f"词汇表现大小:{len(vocab)}")

词汇表现大小:257


In [48]:
print("lesson4 完整 BPE 训练循环")
print("-"*40)

lesson4 完整 BPE 训练循环
----------------------------------------


In [49]:
#重新初始化
vocab = {i:bytes([i]) for i in range(256)}
sequences = [
    tuple(bytes([b]) for b in word.encode("utf-8"))
    for word in ["hello","hello","world","hell","hello"]
]

In [50]:
sequences

[(b'h', b'e', b'l', b'l', b'o'),
 (b'h', b'e', b'l', b'l', b'o'),
 (b'w', b'o', b'r', b'l', b'd'),
 (b'h', b'e', b'l', b'l'),
 (b'h', b'e', b'l', b'l', b'o')]

In [51]:
target_vocab_size=260
merges=[]

In [52]:
print(f"目标词汇量大小:{target_vocab_size}")
print(f"初始词汇表:{len(vocab)}个tokens")

目标词汇量大小:260
初始词汇表:256个tokens


In [53]:
iteration=0
while len(vocab) < target_vocab_size:
    iteration+=1

    #统计pairs频率
    pair_counts=Counter()
    for seq in sequences:
        for i in range(len(seq)-1):
            pair=(seq[i],seq[i+1])
            pair_counts[pair]+=1
    if not pair_counts:
        break

    #选择最频繁的pair
    best_pair = max(pair_counts,key = pair_counts.get)
    max_count=pair_counts[best_pair]

    #创建新的token
    new_token=best_pair[0]+best_pair[1]
    new_id =len(vocab)
    vocab[new_id]=new_token

    #记录合并
    merges.append(best_pair)

    #在序列中合并
    sequences=[merge_in_sequences(seq,best_pair,new_token)for seq in sequences]

    print(f"迭代:{iteration}:")
    print(f"合并:{best_pair[0].decode()}+{best_pair[1].decode()}->{new_token.decode()}")
    print(f"频率：{max_count}次")
    print(f"新 ID:{new_id}")
    print(f"词汇表大小:{len(vocab)}")


迭代:1:
合并:h+e->he
频率：4次
新 ID:256
词汇表大小:257
迭代:2:
合并:he+l->hel
频率：4次
新 ID:257
词汇表大小:258
迭代:3:
合并:hel+l->hell
频率：4次
新 ID:258
词汇表大小:259
迭代:4:
合并:hell+o->hello
频率：3次
新 ID:259
词汇表大小:260


In [54]:
print("\n"+"="*60)
print("最终结果:")
print(f"词汇表大小:{len(vocab)}")
print(f"\n 合并规则:")
for i,(p1,p2) in enumerate(merges):
    print(f" {i+1}.{p1.decode()}+{p2.decode()}->{(p1+p2).decode()}")



最终结果:
词汇表大小:260

 合并规则:
 1.h+e->he
 2.he+l->hel
 3.hel+l->hell
 4.hell+o->hello


In [55]:
print("最终序列:")
for seq in sequences:
    decoded=b''.join(seq).decode("utf-8")
    print(f"{list(seq)}->'{decoded}'")

最终序列:
[b'hello']->'hello'
[b'hello']->'hello'
[b'w', b'o', b'r', b'l', b'd']->'world'
[b'hell']->'hell'
[b'hello']->'hello'


In [56]:
sequences

[(b'hello',),
 (b'hello',),
 (b'w', b'o', b'r', b'l', b'd'),
 (b'hell',),
 (b'hello',)]

In [57]:
print("lesson5 为什么要用字节级别?")
print("-"*40)

lesson5 为什么要用字节级别?
----------------------------------------


In [58]:
print("Q1:中文字符怎么办？")
chinese="你好"
chinese_bytes = chinese.encode("utf-8")
print(f" '{chinese}'->{chinese_bytes}")
print(f"  占用{len(chinese_bytes)}字节")


Q1:中文字符怎么办？
 '你好'->b'\xe4\xbd\xa0\xe5\xa5\xbd'
  占用6字节


In [59]:
print("Q2:emoji怎么办?")
emoji="😅"
emoji_bytes=emoji.encode("utf-8")
print(f" '{emoji}'->{emoji_bytes}")
print(f"  占用{len(emoji_bytes)}字节")


Q2:emoji怎么办?
 '😅'->b'\xf0\x9f\x98\x85'
  占用4字节


In [60]:
print(" 字节级别的优势:")
print("  - 统一处理所有语言")
print("  - 不需要预先知道词汇表")
print("  - 可以处理任意 Unicode 字符")

 字节级别的优势:
  - 统一处理所有语言
  - 不需要预先知道词汇表
  - 可以处理任意 Unicode 字符


In [61]:
print("lesson6 理解增量更新优化")
print("-"*40)

lesson6 理解增量更新优化
----------------------------------------


In [62]:
print("朴素方法(慢):每次合并都重新统计所有pair频率,时间复杂度O(merges * corpus_size)")

朴素方法(慢):每次合并都重新统计所有pair频率,时间复杂度O(merges * corpus_size)


In [63]:
print("优化方法(快):只更新受影响的pairs,例如合并(e,r)->er")
print(" -删除:(h,e),(e,r),(r,ing)")
print(" -添加:(h,er),(er,ing)")
print(" -其他pairs不变")


优化方法(快):只更新受影响的pairs,例如合并(e,r)->er
 -删除:(h,e),(e,r),(r,ing)
 -添加:(h,er),(er,ing)
 -其他pairs不变


#### Problem: (ecoding) BPE Tokenizer Training

In [64]:
import os
import re
from collections import defaultdict
from typing import List,Tuple,Dict
import regex

In [65]:
#GPT-2 预分词
GPT2_PATTERN = regex.compile(
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    )


In [68]:
def run_train_bpe(
    input_path:str | os.PathLike,
    vocab_size:int,
    special_tokens:list[str],
    **kwargs,
)-> tuple[dict[int,bytes],list[tuple[bytes,bytes]]]:
    #step1:参数校验
    if not isinstance(vocab_size,int) or vocab_size<=0:
        raise ValueError("vocab_size 必须是正整数")
    #step2初始化词汇表
    vocab: Dict[int,bytes]= {i:bytes([i]) for i in range(256)}
    next_token_id :int =256

    existing_bytes_values=set(vocab.values())

    for st_str in special_tokens:
        if len(vocab)>= vocab_size:
            break
        st_bytes=st_str.encode("utf-8")
        if st_bytes not in existing_bytes_values:
            vocab[next_token_id]=st_bytes
            existing_bytes_values.add(st_bytes)
            next_token_id+=1
    #step3:读取训练预料
    try :
        with open(input_path,"r",encoding="utf-8",errors="ignore") as f:
            text = f.read()
    except FileNotFoundError:
        text="" #文件不存在时视为空文本
    #step4:预处理和预分词
    ##4.1 按特殊tokens分割
    if special_tokens:
        split_pattern = "|".join(re.escape(tok) for tok in special_tokens)
        chunks = regex.split(split_pattern,text)
    else:
        chunks=[text]
    ##4.2 对每个chunk进行预分词
    token_frequency_table = defaultdict(int)
    for chunk in chunks:
        for word in regex.findall(GPT2_PATTERN,chunk):
            word_bytes = word.encode("utf-8")
            byte_tuple=tuple(bytes([b]) for b in word_bytes)
            token_frequency_table[byte_tuple]+=1
    #step5:初始化 pair 频率统计
    pair_counts=defaultdict(int)
    for byte_tuple,freq in token_frequency_table.items():
        for i in range(len(byte_tuple)-1):
            pair=(byte_tuple[i],byte_tuple[i+1])
            pair_counts[pair]+=freq
    #step6: BPE迭代合并
    merges:List[Tuple[bytes,bytes]]=[]
    while len(vocab)<vocab_size:
        #检查终止条件:
        if not pair_counts:
            break
        #选择最频繁的pair
        max_count=max(pair_counts.values())
        candidates=[pair for pair,count in pair_counts.items() if count == max_count]
        best_pair=max(candidates)
        #创建新token
        new_token_bytes=best_pair[0]+best_pair[1]
        vocab[next_token_id]=new_token_bytes
        next_token_id+=1
        merges.append(best_pair)
        #增量更新pair_counts(性能优化的关键)
        affected_tokens=[]
        for byte_tuple,freq in token_frequency_table.items():
            has_pair = False
            for i in range(len(byte_tuple)-1):
                if byte_tuple[i] == best_pair[0] and byte_tuple[i+1] == best_pair[1]:
                    has_pair = True
                    break
            if has_pair:
                affected_tokens.append((byte_tuple,freq))
        for byte_tuple,freq in affected_tokens:
             for i in range(len(byte_tuple)-1):
                old_pair = (byte_tuple[i],byte_tuple[i+1])
                pair_counts[old_pair]-= freq
                if pair_counts[old_pair] <=0:                        
                    del pair_counts[old_pair]
             new_byte_tuple = _merge_pair_in_sequence(byte_tuple,best_pair,new_token_bytes)
             for i in range(len(new_byte_tuple)-1):
                new_pair=(new_byte_tuple[i],new_byte_tuple[i+1])
                pair_counts[new_pair]+= freq
             del token_frequency_table[byte_tuple]
             token_frequency_table[new_byte_tuple]+= freq
    return vocab,merges


def _merge_pair_in_sequence(
    byte_sequence:tuple[bytes,...],
    pair:tuple[bytes,bytes],
    new_token:bytes,
)-> tuple[bytes,...]:
    new_sequence=[]
    i=0
    while i < len(byte_sequence):
        if i < len(byte_sequence)-1 and byte_sequence[i] == pair[0] and byte_sequence[i+1] == pair[1]:
            new_sequence.append(new_token)
            i+=2
        else:
            new_sequence.append(byte_sequence[i])
            i+=1
    return tuple(new_sequence)

def save_vocab_and_merges(
        vocab:dict[int,bytes],
        merges:list[tuple[bytes,bytes]],
        vocab_path:str,
        merges_path:str,
):  
    import json
    vocab_str={
        token_id:token_bytes.decode("utf-8",errors="replace")
        for token_id,token_bytes in vocab.items()
    }
    with open(vocab_path,"w",encoding="utf-8") as f:
        json.dump(vocab_str, f, ensure_ascii=False, indent=2)
    with open(merges_path,"w",encoding="utf-8") as f:
        for p1,p2 in merges:
            p1_str=p1.decode("utf-8",errors="replace")
            p2_str=p2.decode("utf-8",errors="replace")
            f.write(f"{p1_str} {p2_str}\n")



#### Problem: BPE training on TinyStories

In [2]:
%cd /Users/apple/Desktop/cs336/assignment1-basics-main/homework

/Users/apple/Desktop/cs336/assignment1-basics-main/homework


In [3]:
from BpeTrainingOnTinystories import main
main()

BPE Training on TinyStories Dataset
输入文件: /Users/apple/Desktop/cs336/assignment1-basics-main/data/TinyStories-train.txt
目标词汇表大小: 10,000
特殊tokens: ['<|endoftext|>']

📊 输入文件信息:
   文件大小: 1.79 GB

🚀 开始训练...

✅ 训练成功完成!
📈 训练统计:
   训练时间: 374.09 秒 (6.23 分钟, 0.10 小时)
   内存使用: 309.23 MB
   内存增长: 217.80 MB
   词汇表大小: 10,000 个tokens
   合并次数: 9,743 次

💾 保存结果到磁盘...
   ✓ 词汇表已保存到: /Users/apple/Desktop/cs336/assignment1-basics-main/data/tinystories_vocab.json
   ✓ 合并规则已保存到: /Users/apple/Desktop/cs336/assignment1-basics-main/data/tinystories_merges.txt

🔍 词汇表分析:
   前10个最长的tokens:
      1. ID= 6413, 长度= 15 bytes
         内容:  accomplishment
      2. ID= 8028, 长度= 15 bytes
         内容:  responsibility
      3. ID= 9036, 长度= 15 bytes
         内容:  disappointment
      4. ID= 9686, 长度= 15 bytes
         内容:  recommendation
      5. ID= 3604, 长度= 14 bytes
         内容:  uncomfortable
      6. ID= 3816, 长度= 14 bytes
         内容:  compassionate
      7. ID= 4851, 长度= 14 bytes
         内容:  understanding
      8.

1.在TinyStories 数据集上训练一个字节级别的 BPE 分词器，使用最大词汇表大小10,000。确保将 TinyStories 的特殊 token <｜end▁of▁sentence｜> 添加到词汇表中。将生成的词汇表和合并规则序列化到磁盘以供后续检查。
问：1）训练花了多少小时和内存？2）词汇表中最长的token是什么？3）他合理吗？

In [4]:
from BpeTrainingOnTinystories import performance_analysis
performance_analysis()


🔬 性能分析模式

✅ 性能分析训练完成:
   词汇表大小: 10,000
   合并次数: 9,743

📊 性能分析结果 (按累计时间排序, 前30名):
----------------------------------------------------------------------
         3546927601 function calls (3546927593 primitive calls) in 706.044 seconds

   Ordered by: cumulative time
   List reduced from 214 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      145  253.439    1.748 1228.148    8.470 /opt/miniconda3/lib/python3.13/asyncio/base_events.py:1970(_run_once)
      145    0.479    0.003  703.329    4.851 /opt/miniconda3/lib/python3.13/selectors.py:540(select)
      144  142.046    0.986  170.829    1.186 {method 'control' of 'select.kqueue' objects}
2353948771  154.507    0.000  154.507    0.000 /Users/apple/Desktop/cs336/assignment1-basics-main/homework/bpe_tokenizer.py:131(<genexpr>)
  2119719    0.844    0.000   81.910    0.000 /opt/miniconda3/lib/python3.13/site-packages/regex/_main.py:331(findall)
  2119719   69.821    0.000   69.821

1）374.09;309.23 MB。2）accomplishment 。3）合理。童话故事的受众群主要是小孩，词汇量相对有限，不会超过10k。accomplishment本身很长，一旦在文章中出现，可能会成为最长的单词。

2.分析你的代码。分词器训练过程中哪个部分耗时最多

byte_tuple = tuple(bytes([b]) for b in word_bytes）。 主要是为每个字节创建单独的bytes对象

#### problem:: BPE Training on OpenWebText

1.在 OpenWebText 数据集上训练一个字节级别的 BPE 分词器，最大词汇表大小为 32,000。将生成的词汇表和合并规则序列化到磁盘以供后续检查。
问：1）词汇表中最长的 token 是什么？这合理吗？2）比较和对比在 TinyStories 与 OpenWebText 上训练得到的分词器

In [3]:
from train_bpe_owt import main
main()

BPE Training on OpenWebText Dataset
输入文件: /Users/apple/Desktop/cs336/assignment1-basics-main/data/owt_train.txt
目标词汇表大小: 32,000
特殊tokens: ['<|endoftext|>']

📊 输入文件信息:
   文件大小: 11.10 GB

🚀 开始训练...

✅ 训练成功完成!
📈 训练统计:
   训练时间: 57659.27 秒 (960.99 分钟, 16.02 小时)
   内存使用: 2019.69 MB (1.97 GB)
   内存增长: 1948.91 MB
   词汇表大小: 32,000 个tokens
   合并次数: 31,743 次

💾 保存结果到磁盘...
   ✓ 词汇表已保存到: /Users/apple/Desktop/cs336/assignment1-basics-main/data/owt_vocab.json
   ✓ 合并规则已保存到: /Users/apple/Desktop/cs336/assignment1-basics-main/data/owt_merges.txt

🔍 词汇表分析:
   前10个最长的tokens:
      1. ID=25822, 长度= 64 bytes
         内容: ÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ
      2. ID=25836, 长度= 64 bytes
         内容: ----------------------------------------------------------------
      3. ID=31274, 长度= 48 bytes
         内容: ————————————————
      4. ID=10900, 长度= 32 bytes
         内容: --------------------------------
      5. ID=15947, 长度= 32 bytes
         内容: ________________________________
      6. ID=16885, 长度= 32 bytes

## 2.6 BPE Tokenizer

In [1]:
print("=" * 60)
print("Tokenizer 分词器 - 互动学习")
print("=" * 60)

Tokenizer 分词器 - 互动学习


In [2]:
print("\n【第1课】理解词汇表 (Vocabulary)")
print("-" * 40)
#一个简单的词汇表示例
vocab = {
    0:b'a',
    1:b'b',
    2:b'c',
    3:b'ab',
    4:b'abc',
}
print("词汇表示例:")
for token_id,token_bytes in vocab.items():
    print(f" ID{token_id}:{token_bytes}-> '{token_bytes.decode()}'")
print("\n💡 关键概念:")
print("  - 词汇表 = Token ID ↔ Token Bytes 的映射")
print("  - 训练 BPE 后，我们会得到一个词汇表")
print("  - 词汇表包含：基础字节 + 合并产生的新 tokens")



【第1课】理解词汇表 (Vocabulary)
----------------------------------------
词汇表示例:
 ID0:b'a'-> 'a'
 ID1:b'b'-> 'b'
 ID2:b'c'-> 'c'
 ID3:b'ab'-> 'ab'
 ID4:b'abc'-> 'abc'

💡 关键概念:
  - 词汇表 = Token ID ↔ Token Bytes 的映射
  - 训练 BPE 后，我们会得到一个词汇表
  - 词汇表包含：基础字节 + 合并产生的新 tokens


In [3]:
print("\n\n【第2课】理解合并规则 (Merges)")
print("-" * 40)
#合并规则示例
merges = [
    (b'a',b'b'),
    (b'ab',b'c'),
]

print("合并规则示例(按顺序):")
for i,(p1,p2) in enumerate(merges):
    merged = p1 + p2
    print(f" {i+1}.{p1.decode()}+{p2.decode()}-> {merged.decode()}")

print("\n💡 关键概念:")
print("  - 合并规则的顺序很重要！")
print("  - 编码时按顺序应用合并规则")
print("  - 先合并的优先级更高")




【第2课】理解合并规则 (Merges)
----------------------------------------
合并规则示例(按顺序):
 1.a+b-> ab
 2.ab+c-> abc

💡 关键概念:
  - 合并规则的顺序很重要！
  - 编码时按顺序应用合并规则
  - 先合并的优先级更高


In [4]:
print("\n\n【第3课】手动模拟 Encode 过程")
print("-" * 40)

text = "abc"
print(f"输入文本：'{text}'")

#step 1:转为字节
text_bytes = text.encode("utf-8")
print(f"Step 1 - 转为字节: {text_bytes}")

#step 2:拆分为单字节 tokens
tokens = [bytes([b]) for b in text_bytes]
print(f"Step 2 - 拆分为单字节 tokens: {tokens}")

#step 3:应用合并规则
print("Step 3 - 应用合并规则:")

#创建 bytes->id 映射
vocab_bytes_to_id = {v :k for k,v in vocab.items()}

iteration = 0
while True:
    iteration += 1
    #查找可以合并的pairs
    mergeable_pairs = []
    for i in range(len(tokens)-1):
        pair = (tokens[i],tokens[i+1])
        if pair in merges:
            #找到这个pair在merges中的位置（优先级）
            priority = merges.index(pair)
            mergeable_pairs.append((priority, i, pair))
    
    if not mergeable_pairs:
        print(f" 迭代{iteration}:没有更多可合并的pairs")
        break

    #选择优先级最高的（在merges中最靠前的）
    mergeable_pairs.sort() #按priority排序
    _, best_pos, best_pair =mergeable_pairs[0]

    #合并
    merged_token = best_pair[0] + best_pair[1]
    print(f" 迭代{iteration}: 在位置{best_pos}合并{best_pair[0].decode()}+{best_pair[1].decode()}->{merged_token.decode()}")

    #执行合并
    new_tokens = tokens[:best_pos] + [merged_token] + tokens[best_pos+2:]
    tokens = new_tokens
    print(f"            最终 tokens: {tokens}")

#Step 4 :转为 IDs:
print(f"Step 4 - 转为 IDs:")
token_ids = [vocab_bytes_to_id[token] for token in tokens if token in vocab_bytes_to_id]
print(f"            最终 IDs: {token_ids}")

print("\n💡 Encode过程总结:")
print(" 1. 文本->字节")
print(" 2. 字节->初始单字节 tokens")
print(" 3. 按合并规则迭代合并 tokens")
print(" 4. 最终 tokens->IDs")




    




【第3课】手动模拟 Encode 过程
----------------------------------------
输入文本：'abc'
Step 1 - 转为字节: b'abc'
Step 2 - 拆分为单字节 tokens: [b'a', b'b', b'c']
Step 3 - 应用合并规则:
 迭代1: 在位置0合并a+b->ab
            最终 tokens: [b'ab', b'c']
 迭代2: 在位置0合并ab+c->abc
            最终 tokens: [b'abc']
 迭代3:没有更多可合并的pairs
Step 4 - 转为 IDs:
            最终 IDs: [4]

💡 Encode过程总结:
 1. 文本->字节
 2. 字节->初始单字节 tokens
 3. 按合并规则迭代合并 tokens
 4. 最终 tokens->IDs


In [6]:
print("\n\n【第4课】理解 Decode 过程")
print("-" * 40)

token_ids = [4] #abc的ID
print(f"输入 token IDs: {token_ids}")

#Step 1: 从词汇表中查找token bytes
print(f"\n Step 1 -查找 token bytes:")
all_bytes=[]
for tid in token_ids:
    if tid in vocab:
        token_bytes = vocab[tid]
        all_bytes.append(token_bytes)
        print(f" ID{tid} -> {token_bytes}")

#Step 2: 合并所有字节
combined=b''.join(all_bytes)
print(f"\n Step 2 - 合并所有字节: {combined}")

#Step 3: 解码为文本
decoded_text = combined.decode("utf-8")
print(f"\n Step 3 - 解码为文本: '{decoded_text}'")

print("\n💡 Decode过程总结:")
print(" 1. Token IDs -> Token Bytes(查词汇表)")
print(" 2. 合并所有 token bytes")
print(" 3. bytes -> 文本(utf-8解码)")




【第4课】理解 Decode 过程
----------------------------------------
输入 token IDs: [4]

 Step 1 -查找 token bytes:
 ID4 -> b'abc'

 Step 2 - 合并所有字节: b'abc'

 Step 3 - 解码为文本: 'abc'

💡 Decode过程总结:
 1. Token IDs -> Token Bytes(查词汇表)
 2. 合并所有 token bytes
 3. bytes -> 文本(utf-8解码)


In [8]:
print("\n\n【第5课】理解预分词 (Pre-tokenization)")
print("-" * 40)

text = "Hello, world!"
print(f"输入: '{text}'")

# GPT-2 预分词模式
import regex
GPT2_PATTERN = regex.compile(
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)
pre_tokens = regex.findall(GPT2_PATTERN,text)
print(f"预分词结果:")
for i,token in enumerate(pre_tokens):
    print(f"{i+1}: '{token}'")

print("\n💡 为什么需要预分词:")
print(" - 把文本分成语言学单元")
print(" - 保持单词边界，避免跨单词合并")
print(" - 例如：‘hello'和‘world’ 分别处理，不会产生‘helloworld’")





【第5课】理解预分词 (Pre-tokenization)
----------------------------------------
输入: 'Hello, world!'
预分词结果:
1: 'Hello'
2: ','
3: ' world'
4: '!'

💡 为什么需要预分词:
 - 把文本分成语言学单元
 - 保持单词边界，避免跨单词合并
 - 例如：‘hello'和‘world’ 分别处理，不会产生‘helloworld’


In [10]:
print("\n\n【第6课】处理特殊 Tokens (Special Tokens)")
print("-" * 40)

# 扩展词汇表，添加特殊 tokens
special_tokens = ["<eos>", "<pad>"]
print(f"特殊 tokens: {special_tokens}")

# 添加到词汇表
vocab_with_special = vocab.copy()
next_id = max(vocab_with_special.keys()) + 1

for st in special_tokens:
    vocab_with_special[next_id] = st.encode("utf-8")
    print(f"  添加 {st} → ID {next_id}")
    next_id += 1

print("\n💡 特殊 tokens 的特点:")
print("  - 不会被拆分")
print("  - 始终作为单个 token")
print("  - 常用: <eos>(结束), <pad>(填充), <unk>(未知)")





【第6课】处理特殊 Tokens (Special Tokens)
----------------------------------------
特殊 tokens: ['<eos>', '<pad>']
  添加 <eos> → ID 5
  添加 <pad> → ID 6

💡 特殊 tokens 的特点:
  - 不会被拆分
  - 始终作为单个 token
  - 常用: <eos>(结束), <pad>(填充), <unk>(未知)


In [14]:
print("\n\n【第7课】完整的 Encode/Decode 示例")
print("-" * 40)

# 使用一个更真实的词汇表
print("创建一个简单的词汇表...")

# 基础字节 (简化版，只显示部分)
simple_vocab = {i: bytes([i]) for i in range(256)}

# 添加一些合并后的 tokens
simple_vocab[256] = b'he'
simple_vocab[257] = b'll'
simple_vocab[258] = b'o'
simple_vocab[259] = b'hello'

vocab_bytes_to_id ={v: k for k,v in simple_vocab.items()}
merges = [
    (b'h', b'e'),  # he
    (b'l', b'l'),  # ll
]

print(f"词汇表包含（部分）:")
for tid in [ord('h'), ord('e'), ord('l'), ord('o'), 256, 257, 259]:
    token_bytes = simple_vocab[tid]
    print(f" ID{tid}: {token_bytes} -> '{token_bytes.decode('utf-8',errors='replace')}'")

#Encode "hello"
text = "hello"
print(f"\n Encode: '{text}'")

#预分词
word_bytes = text.encode("utf-8")
tokens = [bytes([b]) for b in word_bytes]
print(f" 初始 tokens: {[t.decode() for t in tokens]}")

#应用合并
def apply_bpe_merge(tokens,merges,vocab):
    """应用bpe合并"""
    while True:
        #找到优先级最高的可合并的pair
        best_pair = None
        best_pos =-1
        best_priority = len(merges) #越小优先级越高

        for i in range(len(tokens)-1):
            pair = (tokens[i],tokens[i+1])
            if pair in merges:
                priority = merges.index(pair)
                if priority < best_priority:
                    best_pair = pair
                    best_pos = i
                    best_priority = priority
        if best_pair is None:
            break

        #合并
        merged = best_pair[0] + best_pair[1]
        tokens = tokens[:best_pos] + [merged] + tokens[best_pos+2:]

    return tokens

tokens = apply_bpe_merge(tokens,merges,simple_vocab)
print(f" 合并后 tokens: {[t.decode() for t in tokens]}")

#转化为IDs
token_ids=[vocab_bytes_to_id[t] for t in tokens]
print(f" Token IDs: {token_ids}")

#Decode
print(f" Decode: {token_ids}")
all_bytes = [simple_vocab[tid] for tid in token_ids]
combined = b"".join(all_bytes)
decoded = combined.decode("utf-8")
print(f" 解码结果: '{decoded}'")

print("\n✅ Roundtrip 成功!")




【第7课】完整的 Encode/Decode 示例
----------------------------------------
创建一个简单的词汇表...
词汇表包含（部分）:
 ID104: b'h' -> 'h'
 ID101: b'e' -> 'e'
 ID108: b'l' -> 'l'
 ID111: b'o' -> 'o'
 ID256: b'he' -> 'he'
 ID257: b'll' -> 'll'
 ID259: b'hello' -> 'hello'

 Encode: 'hello'
 初始 tokens: ['h', 'e', 'l', 'l', 'o']
 合并后 tokens: ['he', 'll', 'o']
 Token IDs: [256, 257, 258]
 Decode: [256, 257, 258]
 解码结果: 'hello'

✅ Roundtrip 成功!


In [16]:
print("\n\n【第8课】流式编码大文件 (Memory Efficient)")
print("-" * 40)

print("假设我们有一个 1GB 的文本文件:")
print("  ❌ 错误方法: 全部读入内存 → encode()")
print("              需要: 1GB + token_ids 内存")
print("")
print("  ✅ 正确方法: 逐行读取 → encode_iterable()")
print("              需要: 1 行文本 + 当前 token_ids 内存")
# 模拟流式处理
def mock_file_lines():
    """模拟文件行"""
    yield "hello world\n"
    yield "this is a test\n"
    yield "end of file\n"

print("\n示例代码:")
print("  with open('large_file.txt') as f:")
print("      for token_id in tokenizer.encode_iterable(f):")
print("          # 处理 token_id")
print("          pass")

print("\n💡 encode_iterable 的优势:")
print("  - 不会一次性加载整个文件")
print("  - 逐行 yield token IDs")
print("  - 内存占用 = O(单行大小)")



【第8课】流式编码大文件 (Memory Efficient)
----------------------------------------
假设我们有一个 1GB 的文本文件:
  ❌ 错误方法: 全部读入内存 → encode()
              需要: 1GB + token_ids 内存

  ✅ 正确方法: 逐行读取 → encode_iterable()
              需要: 1 行文本 + 当前 token_ids 内存

示例代码:
  with open('large_file.txt') as f:
      for token_id in tokenizer.encode_iterable(f):
          # 处理 token_id
          pass

💡 encode_iterable 的优势:
  - 不会一次性加载整个文件
  - 逐行 yield token IDs
  - 内存占用 = O(单行大小)


In [17]:
print("\n\n" + "=" * 60)
print("🎓 恭喜！你已经理解了 Tokenizer 的核心概念")
print("=" * 60)

print("\n关键要点:")
print("  ✅ Encode: 文本 → 预分词 → 字节 → BPE合并 → Token IDs")
print("  ✅ Decode: Token IDs → Token Bytes → 合并 → 文本")
print("  ✅ 预分词: 保持单词边界")
print("  ✅ 特殊 tokens: 不拆分，单独处理")
print("  ✅ 流式编码: 内存高效处理大文件")

print("\n下一步建议:")
print("  1. 阅读 bpe_tokenizer.py 的 Tokenizer 类")
print("  2. 对比每个方法和这里的示例")
print("  3. 使用 from_files() 加载 GPT-2 词汇表测试")
print("  4. 运行 pytest tests/test_tokenizer.py 验证实现")




🎓 恭喜！你已经理解了 Tokenizer 的核心概念

关键要点:
  ✅ Encode: 文本 → 预分词 → 字节 → BPE合并 → Token IDs
  ✅ Decode: Token IDs → Token Bytes → 合并 → 文本
  ✅ 预分词: 保持单词边界
  ✅ 特殊 tokens: 不拆分，单独处理
  ✅ 流式编码: 内存高效处理大文件

下一步建议:
  1. 阅读 bpe_tokenizer.py 的 Tokenizer 类
  2. 对比每个方法和这里的示例
  3. 使用 from_files() 加载 GPT-2 词汇表测试
  4. 运行 pytest tests/test_tokenizer.py 验证实现


# Transform LM

In [2]:
import torch
from einops import rearrange, einsum

## Linear 模块实现

In [1]:
import torch
import torch.nn as nn
from torch import Tensor

/Users/apple/Desktop/cs336/.venv/lib/python3.9/site-packages/torch/_subclasses/functional_tensor.py:279: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:81.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [4]:
class Linear(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = nn.Parameter(
            torch.empty((in_features, out_features), device=device, dtype=dtype)
        )
        nn.init.trunc_normal_(self.W)
    
    def forward(self, x: Tensor) -> Tensor:
        return x @ self.W

        

## Embedding 模块实现

In [5]:
class Embedding(nn.Module):
    def __init__(
        self,
        num_embeddings: int,
        embedding_dim: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.weight = nn.Parameter(
            torch.empty((num_embeddings, embedding_dim), device=device, dtype=dtype)
        )
        nn.init.trunc_normal_(self.weight)
    
    def forward(self, token_ids: Tensor) -> Tensor:
        return self.weight[token_ids]


        

    

## RMSnorm 实现

In [6]:
class RMSNorm(nn.Module):
    def __init__(
        self,
        d_model: int,
        eps: float = 1e-5,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(
            torch.ones(d_model, device=device, dtype=dtype)
        )

    def forward(self, x: Tensor) -> Tensor:
        original_dtype = x.dtype
        x_float = x.to(torch.float32)
        variance = x_float.pow(2).mean(dim=-1, keepdim=True)
        x_norm = x_float * torch.rsqrt(variance + self.eps)
        x_norm = x_norm.to(original_dtype)
        return x_norm * self.weight
        


## SwiGLU

In [ ]:
def swiglu_ffn(d_model: int, d_ff: int | None = None) -> int:
    if d_ff is not None:
        return d_ff
    
    target_d_ff = int(8 * d_model / 3)
    d_ff = ((target_d_ff + 63) // 64) * 64
    return d_ff

class SwiGLU(nn.Module):
    def __init__(
        self,
        d_model: int,
        d_ff: int | None = None,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()

        self.d_model = d_model
        self.d_ff = swiglu_ffn(d_model, d_ff)

        # w1: 上投影层 (d_model -> d_ff)
        self.w1 = nn.Linear(d_model, self.d_ff, bias=False, device=device, dtype=dtype)

        # w2: 下投影层 (d_ff -> d_model)
        self.w2 = nn.Linear(self.d_ff, d_model, bias=False, device=device, dtype=dtype)

        # w3: 门控投影层 (d_model -> d_ff)
        self.w3 = nn.Linear(d_model, self.d_ff, bias=False, device=device, dtype=dtype)

        

    

## RoPE

In [1]:
import torch
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
import numpy as np
from einops import rearrange, einsum

In [8]:
!pip install numpy -i https://pypi.tuna.tsinghua.edu.cn/simple


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 5.7 MB/s  0:00:00 eta 0:00:01


In [3]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, theta: float, d_k: int, max_seq_len: int, device=None):
        super().__init__()
        if d_k % 2 != 0:
            raise ValueError(f"d_k must be even, got {d_k}")
        
        self.theta = theta
        self.d_k = d_k
        self.max_seq_len = max_seq_len

        freq_indices = torch.arange(0, d_k, 2, dtype=torch.float32, device=device)
        freqs = 1.0 / (theta ** (freq_indices / d_k))

        positions = torch.arange(max_seq_len, dtype=torch.float32, device=device)

        angles = torch.outer(positions, freqs)

        self.register_buffer("cos_cache", angles.cos(), persistent=False)
        self.register_buffer("sin_cache", angles.sin(), persistent=False)

    def forward(
        self, x: Float[Tensor, "... seq_len d_k"], token_positions: Int[Tensor, "... seq_len"]
    ) -> Float[Tensor, "... seq_len d_k"]:
        cos = self.cos_cache[token_positions]
        sin = self.sin_cache[token_positions]

        x_even = x[..., 0::2]  
        x_odd = x[..., 1::2]

        rotated_even = x_even * cos - x_odd * sin
        rotated_odd = x_even * sin + x_odd * cos

        stacked = torch.stack([rotated_even, rotated_odd], dim=-1)

        return stacked.flatten(-2)



        









## scaled dot product Attention

In [1]:
import torch
from jaxtyping import Bool, Float
from torch import Tensor
import math
from homework.softmax import softmax

def scaled_dot_product_attention(
    Q: Float[Tensor, " ... queries d_k"],
    K: Float[Tensor, " ... keys d_k"],
    V: Float[Tensor, " ... values d_v"],
    mask: Bool[Tensor, " ... queries keys"] | None = None,
) -> Float[Tensor, " ... queries d_v"]:
    
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))

    attn_weights = softmax(scores, dim=-1)

    output = torch.matmul(attn_weights, V)
    return output

## Multihead-attention

In [1]:
import torch
import torch.nn as nn
from jaxtyping import Float, Int
from torch import Tensor
from einops import rearrange
from homework.attention import scaled_dot_product_attention

class MultiHeadSelfAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ):
        super().__init__()

        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads})")
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Per-head dimension
        self.d_v = d_model // num_heads  # Per-head dimension

        self.q_proj = nn.Linear(d_model, d_model, bias=False, device=device, dtype=dtype)
        self.k_proj = nn.Linear(d_model, d_model, bias=False, device=device, dtype=dtype)
        self.v_proj = nn.Linear(d_model, d_model, bias=False, device=device, dtype=dtype)

        self.output_proj = nn.Linear(d_model, d_model, bias=False, device=device, dtype=dtype)

    def forward(
        self,
        x: Float[Tensor, "... seq_len d_model"],
        token_positions: Int[Tensor, "... seq_len"] | None = None,
    ) -> Float[Tensor, "... seq_len d_model"]:
        
        *batch_dims, seq_len, d_model = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = rearrange(Q, "... seq (head d) -> ... head seq d", head=self.num_heads)
        K = rearrange(K, "... seq (head d) -> ... head seq d", head=self.num_heads)
        V = rearrange(V, "... seq (head d) -> ... head seq d", head=self.num_heads)

        causal_mask = torch.tril(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool)
        )

        attn_output = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
        attn_output = rearrange(attn_output, "... head seq d -> ... seq (head d)")

        output = self.output_proj(attn_output)

        return output






    

# Training LM

## Cross Entropy Loss

In [1]:
import torch
from einops import reduce
from jaxtyping import Float, Int
from torch import Tensor

In [2]:
def cross_entropy(
    inputs: Float[Tensor, " batch_size vocab_size"],
    targets: Int[Tensor, " batch_size"],
) -> Float[Tensor, ""]:
    x_max = reduce(inputs, "b v -> b 1", "max")
    logits_stable = inputs - x_max
    log_sum_exp = torch.log(reduce(torch.exp(logits_stable), "b v -> b", "sum"))
    target_logits = logits_stable[torch.arange(inputs.shape[0], device=inputs.device), targets]
    losses = -target_logits + log_sum_exp
    return reduce(losses, "b -> ", "mean")

    

#### Problem: learning_rate_tuning

In [ ]:
import torch
import math
from collections.abc import Callable
from typing import Optional

class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                t = state.get("t", 0)
                grad = p.grad.data
                p.data -= lr / math.sqrt(t + 1) * grad
                state["t"] = t + 1
        return loss

#对比不同learning rate: 1e1, 1e2, 1e3
learning_rates = [1e1, 1e2, 1e3]
n_iters = 10

print("=" * 60)
print("Learning Rate Tuning 实验")
print("=" * 60)

for lr in learning_rates:
    weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
    opt = SGD([weights], lr=lr)
    losses = []

    for t in range(n_iters):
        opt.zero_grad()
        loss = (weights ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        opt.step()

    trend = "收敛" if losses[-1] < losses[0] else "发散"
    print(f"\nlr={lr:.0e}: {trend}")
    for t, l in enumerate(losses):
        bar = "#" * int(min(l, 50))
        print(f"  iter {t:2d}: loss={l:.4f} {bar}")

lr=1e1: loss衰减最快，每步大幅下降，10步后接近0。lr=1e2: loss先快速下降但随后开始振荡/发散，步长过大导致越过最优点来回跳。lr=1e3: loss直接爆炸发散，梯度更新步长远超合理范围，参数被推到极端值。

#### Problem: AdamW

In [ ]:
import torch
import math
from collections.abc import Callable
from typing import Optional

class AdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr, "betas": betas, "eps": eps, "weight_decay": weight_decay}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            weight_decay = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad.data
                state = self.state[p]

                #初始化状态
                if len(state) == 0:
                    state["t"] = 0
                    state["m"] = torch.zeros_like(p.data)  #一阶矩(均值)
                    state["v"] = torch.zeros_like(p.data)  #二阶矩(方差)

                state["t"] += 1
                t = state["t"]
                m, v = state["m"], state["v"]

                #更新矩估计
                # m <- beta1*m + (1-beta1)*g
                m.mul_(beta1).add_(grad, alpha=1 - beta1)
                # v <- beta2*v + (1-beta2)*g^2
                v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                #偏差校正后的学习率: alpha_t = lr * sqrt(1-beta2^t) / (1-beta1^t)
                alpha_t = lr * math.sqrt(1 - beta2 ** t) / (1 - beta1 ** t)

                #更新参数: theta <- theta - alpha_t * m / (sqrt(v) + eps)
                p.data.addcdiv_(m, v.sqrt().add_(eps), value=-alpha_t)

                #解耦weight decay: theta <- theta - lr*lambda*theta
                p.data.add_(p.data, alpha=-lr * weight_decay)

        return loss

#验证: 与PyTorch AdamW对比
torch.manual_seed(42)
model_ours = torch.nn.Linear(3, 2, bias=False)
torch.manual_seed(42)
model_ref = torch.nn.Linear(3, 2, bias=False)

opt_ours = AdamW(model_ours.parameters(), lr=1e-3, weight_decay=0.01)
opt_ref = torch.optim.AdamW(model_ref.parameters(), lr=1e-3, weight_decay=0.01)

for _ in range(100):
    x = torch.rand(3)
    for model, opt in [(model_ours, opt_ours), (model_ref, opt_ref)]:
        opt.zero_grad()
        loss = ((model(x) - torch.tensor([x[0]+x[1], -x[2]])) ** 2).sum()
        loss.backward()
        opt.step()

diff = (model_ours.weight.data - model_ref.weight.data).abs().max().item()
print(f"与PyTorch AdamW最大差异: {diff:.2e}")
print(f"匹配: {'YES' if diff < 1e-4 else 'NO'}")

#### Problem: AdamW 资源核算 (adamwAccounting)

**题目翻译：**

计算使用 AdamW 训练所需的内存和计算量。假设所有张量使用 float32。

**(a)** 运行 AdamW 需要多少**峰值内存**？将答案按**参数、激活值、梯度、优化器状态**分解。用 `batch_size` 和模型超参数（`vocab_size`, `context_length`, `num_layers`, `d_model`, `num_heads`）表示。假设 $d_{ff} = 4 \times d_{model}$。

计算激活值内存时，只考虑以下组件：
- **Transformer block**：RMSNorm(s)、多头自注意力子层（QKV投影、$Q^\top K$ 矩阵乘法、softmax、值的加权和、输出投影）、前馈网络（W1矩阵乘法、SiLU、W2矩阵乘法）
- **最终 RMSNorm**
- **输出嵌入层**
- **logits 上的交叉熵**

**(b)** 用 GPT-2 XL 的参数代入，得到只依赖 `batch_size` 的表达式。在 80GB 内存限制下，最大 batch size 是多少？

**(c)** 运行一步 AdamW 需要多少 FLOPs？

**(d)** 假设 MFU=50%，A100 峰值 19.5 TFLOP/s (float32)，反向传播 FLOPs = 2× 前向传播，训练 GPT-2 XL 400K 步、batch_size=1024 需要多少天？

##### (a) 峰值内存分析

**符号定义：**
- $B$ = batch_size, $V$ = vocab_size, $T$ = context_length
- $L$ = num_layers, $d$ = d_model, $H$ = num_heads
- $d_{ff} = 4d$, $d_k = d/H$（每头维度）
- 使用 weight tying（输入/输出嵌入共享权重，GPT-2 标准做法）
- 使用 RoPE（无可学习位置嵌入参数）

---

**1. 参数量 $P$**

| 组件 | 参数量 |
|------|--------|
| Token Embedding | $V \cdot d$ |
| 每个 Transformer Block (×L): | |
| &nbsp;&nbsp; QKV 投影 | $3d^2$ |
| &nbsp;&nbsp; 输出投影 | $d^2$ |
| &nbsp;&nbsp; W1 (FFN) | $d \cdot 4d = 4d^2$ |
| &nbsp;&nbsp; W2 (FFN) | $4d \cdot d = 4d^2$ |
| &nbsp;&nbsp; 2× RMSNorm | $2d$ |
| 最终 RMSNorm | $d$ |

$$P = Vd + L(12d^2 + 2d) + d$$

**参数内存** = $4P$ 字节

---

**2. 梯度**

每个参数对应一个梯度，大小相同。

**梯度内存** = $4P$ 字节

---

**3. 优化器状态（AdamW）**

AdamW 为每个参数维护一阶矩 $m$ 和二阶矩 $v$，各占 $4P$ 字节。

**优化器状态内存** = $8P$ 字节

---

**4. 激活值 $A$**

**每个 Transformer Block 的激活值（每层输出张量大小）：**

| 组件 | 元素数 |
|------|--------|
| RMSNorm 1 输出 | $BTd$ |
| Q, K, V | $3BTd$ |
| $Q^\top K$ 注意力分数 | $BHT^2$ |
| Softmax 输出 | $BHT^2$ |
| 加权值求和 | $BTd$ |
| 输出投影 | $BTd$ |
| RMSNorm 2 输出 | $BTd$ |
| W1 输出 | $4BTd$ |
| SiLU 输出 | $4BTd$ |
| W2 输出 | $BTd$ |

每层合计：$16BTd + 2BHT^2$

**Transformer 之外的激活值：**

| 组件 | 元素数 |
|------|--------|
| 最终 RMSNorm 输出 | $BTd$ |
| 输出嵌入 (logits) | $BTV$ |
| 交叉熵 (softmax 概率) | $BTV$ |

$$A = L(16BTd + 2BHT^2) + BTd + 2BTV$$

**激活值内存** = $4A$ 字节

---

**总峰值内存：**

$$\boxed{\text{Memory} = 16P + 4A = 16[Vd + L(12d^2 + 2d) + d] + 4[L(16BTd + 2BHT^2) + BTd + 2BTV] \text{ 字节}}$$

##### (b) GPT-2 XL 实例化

GPT-2 XL 超参数：V=50257, T=1024, L=48, d=1600, H=25

In [6]:
# GPT-2 XL 参数
V, T, L, d, H = 50257, 1024, 48, 1600, 25

# 参数量
P = V * d + L * (12 * d**2 + 2 * d) + d
print(f"参数量 P = {P:,} ({P/1e9:.2f}B)")

# 非激活内存 (参数 + 梯度 + 优化器状态)
mem_non_act = 16 * P  # bytes
print(f"\n参数+梯度+优化器 = 16P = {mem_non_act:,} bytes = {mem_non_act/1e9:.2f} GB")
print(f"  其中: 参数 4P = {4*P/1e9:.2f} GB")
print(f"        梯度 4P = {4*P/1e9:.2f} GB")
print(f"        优化器 8P = {8*P/1e9:.2f} GB")

# 激活值系数 (A = coeff * B)
act_coeff = L * (16 * T * d + 2 * H * T**2) + T * d + 2 * T * V
mem_act_per_B = 4 * act_coeff  # bytes per batch element
print(f"\n激活值内存 = {mem_act_per_B/1e9:.2f} × B GB")

# 总内存
a = mem_act_per_B / 1e9  # GB per batch element
b = mem_non_act / 1e9     # GB constant
print(f"\n总内存 = {a:.2f} × B + {b:.2f} GB")

# 最大 batch size
max_mem_gb = 80
max_B = (max_mem_gb - b) / a
print(f"\n80 GB 限制下: B ≤ {max_B:.2f}")
print(f"最大 batch_size = {int(max_B)}")

参数量 P = 1,555,126,400 (1.56B)

参数+梯度+优化器 = 16P = 24,882,022,400 bytes = 24.88 GB
  其中: 参数 4P = 6.22 GB
        梯度 4P = 6.22 GB
        优化器 8P = 12.44 GB

激活值内存 = 15.52 × B GB

总内存 = 15.52 × B + 24.88 GB

80 GB 限制下: B ≤ 3.55
最大 batch_size = 3


**结论 (b):**

$$\text{Memory} = 15.52 \times B + 24.88 \text{ (GB)}$$

- $a \approx 15.52$ GB/样本（激活值）
- $b \approx 24.88$ GB（参数+梯度+优化器状态）
- **最大 batch_size = 3**（$(80 - 24.88) / 15.52 \approx 3.55$）

##### (c) 一步 AdamW 的 FLOPs

AdamW 优化器更新步骤对每个参数元素执行以下操作：

| 操作 | 每元素 FLOPs |
|------|-------------|
| $m \leftarrow \beta_1 m + (1-\beta_1)g$ | 3（2次乘法+1次加法） |
| $v \leftarrow \beta_2 v + (1-\beta_2)g^2$ | 4（1次平方+2次乘法+1次加法） |
| 偏差校正 $\hat{m}, \hat{v}$ | 2（2次除法，可折入学习率） |
| $\theta \leftarrow \theta - \alpha_t \hat{m}/(\sqrt{\hat{v}}+\epsilon)$ | 4（sqrt+加+除+乘&减） |
| Weight decay: $\theta \leftarrow \theta(1 - \text{lr}\cdot\lambda)$ | 2（1次乘法+1次减法） |

**每个参数约 15 次浮点运算**

$$\boxed{\text{FLOPs}_{\text{AdamW}} \approx 15P = 15[Vd + L(12d^2 + 2d) + d]}$$

注意：这只是优化器更新的 FLOPs，远小于前向/反向传播的 FLOPs（$O(P)$ vs $O(BTP)$）。

##### (d) 训练时间估算

**前向传播 FLOPs（矩阵乘法主导）：**

每个线性层 $Y = X \cdot W$，其中 $X$ 为 $(BT, m)$，$W$ 为 $(m, n)$，FLOPs = $2 \cdot BT \cdot m \cdot n$

每个 Transformer Block:
| 操作 | FLOPs |
|------|-------|
| QKV 投影 | $3 \times 2BTd^2 = 6BTd^2$ |
| $Q^\top K$ | $2BT^2d$ |
| Attn × V | $2BT^2d$ |
| 输出投影 | $2BTd^2$ |
| W1 (FFN) | $2BT \cdot d \cdot 4d = 8BTd^2$ |
| W2 (FFN) | $2BT \cdot 4d \cdot d = 8BTd^2$ |

每层合计：$24BTd^2 + 4BT^2d$

输出嵌入：$2BTVd$

$$F_{\text{forward}} = L(24BTd^2 + 4BT^2d) + 2BTVd$$

**总 FLOPs/步**（前向 + 反向 = 前向 × 3）：
$$F_{\text{step}} = 3F_{\text{forward}} = 3[L(24BTd^2 + 4BT^2d) + 2BTVd]$$

In [7]:
# GPT-2 XL 训练时间计算
V, T, L, d, H = 50257, 1024, 48, 1600, 25
B_train = 1024  # 训练 batch size
steps = 400_000

# 前向 FLOPs/步
per_block = 24 * B_train * T * d**2 + 4 * B_train * T**2 * d
F_forward = L * per_block + 2 * B_train * T * V * d
F_step = 3 * F_forward  # forward + backward(2x forward)

print(f"前向 FLOPs/步: {F_forward:.3e}")
print(f"总 FLOPs/步 (3×前向): {F_step:.3e}")

# 总训练 FLOPs
total_flops = steps * F_step
print(f"\n总训练 FLOPs (400K 步): {total_flops:.3e}")

# A100 有效吞吐
peak_tflops = 19.5  # TFLOP/s
mfu = 0.5
effective_flops = peak_tflops * 1e12 * mfu  # FLOP/s
print(f"A100 有效吞吐 (50% MFU): {effective_flops:.2e} FLOP/s")

# 训练时间
time_seconds = total_flops / effective_flops
time_days = time_seconds / 86400
time_years = time_days / 365
print(f"\n训练时间: {time_seconds:.2e} 秒")
print(f"         = {time_days:,.0f} 天")
print(f"         ≈ {time_years:.1f} 年")

# 每步时间
time_per_step = F_step / effective_flops
print(f"\n每步耗时: {time_per_step:.1f} 秒 ({time_per_step/60:.1f} 分钟)")
tokens_per_step = B_train * T
print(f"每步处理 tokens: {tokens_per_step:,}")
print(f"总 tokens: {steps * tokens_per_step / 1e9:.1f}B")

前向 FLOPs/步: 3.591e+15
总 FLOPs/步 (3×前向): 1.077e+16

总训练 FLOPs (400K 步): 4.309e+21
A100 有效吞吐 (50% MFU): 9.75e+12 FLOP/s

训练时间: 4.42e+08 秒
         = 5,115 天
         ≈ 14.0 年

每步耗时: 1104.9 秒 (18.4 分钟)
每步处理 tokens: 1,048,576
总 tokens: 419.4B


**结论 (d):**

$$\text{训练时间} \approx 5{,}115 \text{ 天} \approx 14 \text{ 年}$$

**理由：**
- 每步处理 $1024 \times 1024 \approx 10^6$ tokens
- 总共处理约 419B tokens
- 每步 FLOPs ≈ $1.08 \times 10^{16}$
- A100 在 50% MFU 下有效吞吐 $9.75 \times 10^{12}$ FLOP/s
- 每步约 18 分钟，400K 步总计约 5,115 天

这解释了为什么大模型训练需要数百甚至数千张 GPU 并行：单卡训练 1.5B 参数模型在实践中是不可行的。

#### Problem: 余弦学习率调度 (learning_rate_schedule)

**题目翻译：**

**4.4 学习率调度**

训练过程中，能使 loss 下降最快的学习率往往会变化。训练 Transformer 时，通常使用**学习率调度**：一开始用较大的学习率加快更新，然后随训练进行逐渐衰减。本题实现 LLaMA [Touvron et al., 2023] 使用的余弦退火调度。

调度器是一个函数，输入当前步数 $t$ 及相关参数（初始/最终学习率等），返回第 $t$ 步应使用的学习率。

余弦退火学习率调度的参数：(i) 当前迭代 $t$，(ii) 最大学习率 $\alpha_{\max}$，(iii) 最小学习率 $\alpha_{\min}$，(iv) 预热步数 $T_w$，(v) 余弦退火步数 $T_c$。

**三个阶段：**

- **(Warm-up)** 若 $t < T_w$：$\alpha_t = \frac{t}{T_w} \alpha_{\max}$
- **(Cosine annealing)** 若 $T_w \le t \le T_c$：$\alpha_t = \alpha_{\min} + \frac{1}{2}\left(1 + \cos\left(\frac{t - T_w}{T_c - T_w}\pi\right)\right)(\alpha_{\max} - \alpha_{\min})$
- **(Post-annealing)** 若 $t > T_c$：$\alpha_t = \alpha_{\min}$

In [ ]:
import math

def lr_cosine_schedule(
    it: int,
    max_learning_rate: float,
    min_learning_rate: float,
    warmup_iters: int,
    cosine_cycle_iters: int,
) -> float:
    """余弦学习率调度（带线性预热）
    
    (Warm-up)        t < T_w:   alpha = (t / T_w) * alpha_max
    (Cosine anneal)  T_w <= t <= T_c:  alpha = alpha_min + 0.5*(1+cos((t-T_w)/(T_c-T_w)*pi))*(alpha_max - alpha_min)
    (Post-anneal)    t > T_c:   alpha = alpha_min
    """
    if it < warmup_iters:
        # 线性预热：从0线性增长到alpha_max
        return (it / warmup_iters) * max_learning_rate
    elif it <= cosine_cycle_iters:
        # 余弦退火：从alpha_max平滑下降到alpha_min
        progress = (it - warmup_iters) / (cosine_cycle_iters - warmup_iters)
        return min_learning_rate + 0.5 * (1 + math.cos(progress * math.pi)) * (max_learning_rate - min_learning_rate)
    else:
        # 退火结束后：保持alpha_min
        return min_learning_rate

# 验证：用测试用例检查
max_lr, min_lr = 1.0, 0.1
T_w, T_c = 7, 21

lrs = [lr_cosine_schedule(t, max_lr, min_lr, T_w, T_c) for t in range(25)]

print("步数 | 学习率    | 阶段")
print("-" * 40)
for t, lr in enumerate(lrs):
    if t < T_w:
        phase = "预热"
    elif t <= T_c:
        phase = "余弦退火"
    else:
        phase = "退火后"
    bar = "#" * int(lr * 30)
    print(f"  {t:2d} | {lr:.4f}   | {phase:6s} {bar}")

**实现要点：**
- **预热阶段** ($t < T_w$)：学习率从 0 线性增长到 $\alpha_{\max}$，让模型在训练初期用小步长稳定参数
- **余弦退火** ($T_w \le t \le T_c$)：用余弦函数从 $\alpha_{\max}$ 平滑衰减到 $\alpha_{\min}$，比线性衰减更温和
- **退火后** ($t > T_c$)：保持最小学习率 $\alpha_{\min}$ 不变

代码已写入 [homework/lr_schedule.py](homework/lr_schedule.py)，测试通过 `uv run pytest -k test_get_lr_cosine_schedule`

#### Problem: gradient_clipping

**梯度裁剪（Gradient Clipping）**：训练中遇到大梯度会导致不稳定。梯度裁剪在每次反向传播后、优化器更新前，限制梯度的 ℓ₂ 范数。

给定所有参数的梯度 $g$，计算 $\|g\|_2$：
- 若 $\|g\|_2 \le M$：不修改
- 若 $\|g\|_2 > M$：将每个梯度乘以 $\frac{M}{\|g\|_2 + \epsilon}$（$\epsilon = 10^{-6}$）

**直觉**：梯度方向不变，只在"步子太大"时等比例缩短。

In [ ]:
import torch
import torch.nn as nn

def gradient_clipping(parameters, max_l2_norm: float) -> None:
    """梯度裁剪：将全局梯度 L2 范数限制在 max_l2_norm 以内（原地修改）"""
    eps = 1e-6
    grads = [p.grad for p in parameters if p.grad is not None]
    if not grads:
        return
    # 全局 L2 范数 = sqrt(所有梯度元素平方和)
    total_norm = torch.sqrt(sum(g.pow(2).sum() for g in grads))
    if total_norm > max_l2_norm:
        scale = max_l2_norm / (total_norm + eps)
        for g in grads:
            g.mul_(scale)

# 演示
torch.manual_seed(42)
params = [nn.Parameter(torch.randn(3, 3)) for _ in range(3)]
loss = sum(p.sum() for p in params)
loss.backward()

norms_before = torch.sqrt(sum(p.grad.pow(2).sum() for p in params)).item()
print(f"裁剪前梯度 L2 范数: {norms_before:.4f}")

gradient_clipping(params, max_l2_norm=0.5)

norms_after = torch.sqrt(sum(p.grad.pow(2).sum() for p in params)).item()
print(f"裁剪后梯度 L2 范数: {norms_after:.4f}")
print(f"目标最大范数: 0.5")

# 5. Training Loop

## 5.1 Data Loader

训练数据是一个 token ID 的长序列 $x = (x_1, \ldots, x_n)$（多个文档拼接，用 `<|endoftext|>` 分隔）。

**Data Loader 的任务**：将长序列切成 mini-batch，每个 batch 包含 $B$ 个长度为 $m$ 的序列及其对应的 next-token 目标。

例如 $B=1, m=3$：
- 输入：$[x_2, x_3, x_4]$
- 目标：$[x_3, x_4, x_5]$（向右平移一位）

**为什么这样做？**
1. 任意 $1 \le i < n-m$ 都是合法起始位置，采样简单
2. 所有序列等长，无需 padding，硬件利用率高
3. 不需要把整个数据集加载到内存

In [1]:
import numpy as np
import torch

def get_batch(dataset, batch_size, context_length, device):
    """从 token 序列中随机采样一个 batch 的输入和目标。
    
    Args:
        dataset: 1D numpy array，token ID 序列
        batch_size: batch 中序列数量
        context_length: 每个序列的长度
        device: PyTorch 设备字符串
    
    Returns:
        (x, y)，shape 均为 (batch_size, context_length)
        x[i] = dataset[start : start + context_length]
        y[i] = dataset[start+1 : start+1 + context_length]
    """
    # 合法起始位置: 0 ~ len(dataset) - context_length - 1
    max_start = len(dataset) - context_length
    starts = np.random.randint(0, max_start, size=batch_size)
    
    x = np.stack([dataset[s : s + context_length] for s in starts])
    y = np.stack([dataset[s + 1 : s + 1 + context_length] for s in starts])
    
    return (
        torch.tensor(x, dtype=torch.long, device=device),
        torch.tensor(y, dtype=torch.long, device=device),
    )

# 演示
dataset = np.arange(100)  # 简单数据: [0, 1, 2, ..., 99]
x, y = get_batch(dataset, batch_size=3, context_length=5, device="cpu")

print("dataset = [0, 1, 2, ..., 99]")
print(f"\nbatch_size=3, context_length=5:")
print(f"x shape: {x.shape}")
print(f"y shape: {y.shape}")
print(f"\n采样结果:")
for i in range(3):
    print(f"  序列 {i}: x={x[i].tolist()}, y={y[i].tolist()}")
    print(f"           y 恰好是 x 右移一位 ✓" if (x[i] + 1 == y[i]).all() else "")

dataset = [0, 1, 2, ..., 99]

batch_size=3, context_length=5:
x shape: torch.Size([3, 5])
y shape: torch.Size([3, 5])

采样结果:
  序列 0: x=[21, 22, 23, 24, 25], y=[22, 23, 24, 25, 26]
           y 恰好是 x 右移一位 ✓
  序列 1: x=[1, 2, 3, 4, 5], y=[2, 3, 4, 5, 6]
           y 恰好是 x 右移一位 ✓
  序列 2: x=[23, 24, 25, 26, 27], y=[24, 25, 26, 27, 28]
           y 恰好是 x 右移一位 ✓


**实现要点：**

1. **随机采样起始位置**：`np.random.randint(0, len(dataset) - context_length, size=batch_size)`
   - 起始范围 `[0, n - m)`，保证切片 `[start : start + m + 1]` 不越界（需要多一个位置给 target）
2. **输入 x 和目标 y 相差一个位置**：`y = dataset[start+1 : start+1+m]`
   - 这就是 next-token prediction 的本质
3. **放到指定设备**：`torch.tensor(..., device=device)` 让数据直接在 GPU/MPS 上

代码已写入 [homework/data_loading.py](homework/data_loading.py)，测试通过 `uv run pytest -k test_get_batch`